# Rule-based vs ML: comparison on the same test set (v3 dataset)

We compare the **full** rule-based engine (`fraud_checks/services.py`,
all 6 signals) against the ML model on the same test set (v3).
Rules 1–6 cover: account_blocked, insufficient_balance,
limit_exceeded (daily > 200k), high_amount (> 100k),
new_account (< 7d), high_frequency (> 10tx/h).
ML uses threshold 0.64 (tuned on val set, see rf_tuning.ipynb).

In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))
from ml.synthetic.build_dataset import generate_accounts

DATASET_PATH = "../datasets/synthetic_v4.csv"
MODEL_PATH = "../models/fraud_model_v4.pkl"
FEATURE_COLS = ["amount", "account_age_days", "tx_last_hour", "transaction_hour",
              "receiver_tx_count_24h", "sender_daily_amount_sum",
              "amount_to_balance_ratio", "days_since_last_tx"]
BEST_THRESHOLD = 0.64

df = pd.read_csv(DATASET_PATH, parse_dates=["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)

# Re-generate accounts (deterministic, SEED=42) for balance info
from ml.synthetic import config
accounts_df = generate_accounts(config.N_ACCOUNTS, np.random.default_rng(config.SEED))

# same time-based split as in train_model.py (train_ratio=0.8)
split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:].copy()

model = joblib.load(MODEL_PATH)
print(f"Test set: {len(test_df)} rows ({test_df["is_fraud"].mean():.2%} fraud)")

Test set: 4000 rows (3.12% fraud)


## Step 1 — ML model predictions

Using tuned threshold from rf_tuning.ipynb (0.64 instead of default 0.5).

In [3]:
X_test = test_df[FEATURE_COLS]
test_df["ml_proba"] = model.predict_proba(X_test)[:, 1]
test_df["ml_pred"] = (test_df["ml_proba"] >= BEST_THRESHOLD).astype(int)

## Step 2 — Rule-based engine (full)

Logic ported from `fraud_checks/services.py` `calculate_risk()`.
All 6 signals are computed from DataFrame columns:
- account_blocked: is_blocked (False for all synthetic)
- insufficient_balance: amount > sender_balance
- limit_exceeded: sender_daily_amount_sum + amount > 200k
- high_amount: amount > 100k
- high_frequency: tx_last_hour > 10
- new_account: account_age_days < 7

In [4]:
def rule_based_full(row):
    risk_score = 0
    reasons = []
    if row["account_age_days"] < 7:
        risk_score += 20
        reasons.append("new_account")
    if row["amount"] > 100000:
        risk_score += 40
        reasons.append("high_amount")
    if row["tx_last_hour"] > 10:
        risk_score += 30
        reasons.append("high_frequency")
    if row["sender_daily_amount_sum"] + row["amount"] > 200000:
        risk_score += 60
        reasons.append("limit_exceeded")
    if row["amount"] > row["sender_balance_before"]:
        risk_score += 80
        reasons.append("insufficient_balance")
    # account_blocked: is_blocked is False for all synthetic, never fires

    if risk_score >= 60:
        decision = "BLOCKED"
    elif risk_score >= 30:
        decision = "REVIEW"
    else:
        decision = "APPROVED"
    return risk_score, decision, reasons

# Add sender balance column for insufficient_balance rule
balance_map = accounts_df.set_index("account_id")["balance"]
test_df["sender_balance_before"] = test_df["sender_account_id"].map(balance_map)

results = test_df.apply(rule_based_full, axis=1, result_type="expand")
test_df["rule_risk_score"] = results[0]
test_df["rule_decision"] = results[1]
test_df["rule_reasons"] = results[2]
# REVIEW and BLOCKED count as "system flagged something" -> 1
test_df["rule_pred"] = test_df["rule_decision"].isin(["BLOCKED", "REVIEW"]).astype(int)

## Step 3 — Metric comparison (full rules vs ML, with recall by pattern)

In [5]:
print("=== Rule-based (full) ===\n")
print(classification_report(test_df["is_fraud"], test_df["rule_pred"],
                              target_names=["normal", "fraud"]))

print("\n=== ML model ===\n")
print(classification_report(test_df["is_fraud"], test_df["ml_pred"],
                              target_names=["normal", "fraud"]))

# --- Recall by fraud_pattern ---
print("\n=== Recall by fraud_pattern ===\n")
for system_name, pred_col in [("Rules", "rule_pred"), ("ML", "ml_pred")]:
    print(f"--- {system_name} ---")
    for pattern in sorted(test_df["fraud_pattern"].dropna().unique()):
        mask = test_df["fraud_pattern"] == pattern
        total = int(mask.sum())
        caught = int(((test_df.loc[mask, "is_fraud"] == 1) & (test_df.loc[mask, pred_col] == 1)).sum())
        recall = caught / total if total > 0 else 0
        print(f"  {pattern:35s}: {caught:3d}/{total}  (recall={recall:.2%})")
    print()

# --- Feature importances ---
print("=== Feature importances ===\n")
importances = pd.Series(
    model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)
print(importances)

=== Rule-based (full) ===

              precision    recall  f1-score   support

      normal       0.98      0.81      0.89      3875
       fraud       0.07      0.42      0.12       125

    accuracy                           0.80      4000
   macro avg       0.52      0.62      0.50      4000
weighted avg       0.95      0.80      0.86      4000


=== ML model ===

              precision    recall  f1-score   support

      normal       0.99      0.99      0.99      3875
       fraud       0.80      0.69      0.74       125

    accuracy                           0.98      4000
   macro avg       0.90      0.84      0.87      4000
weighted avg       0.98      0.98      0.98      4000


=== Recall by fraud_pattern ===

--- Rules ---
  balance_drain_fraud                :   0/19  (recall=0.00%)
  dormant_reactivation_fraud         :   6/8  (recall=75.00%)
  mule_fraud                         :  14/60  (recall=23.33%)
  new_account_fraud                  :   4/5  (recall=80.00%)
  s

## Step 4 — Where they disagree

The most informative part: examining specific cases of disagreement.

In [6]:
# ML catches, rule misses (ML better here)
ml_catches_rule_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 1)
]
print(f"ML caught fraud that rules missed: {len(ml_catches_rule_misses)}")
disp_cols = ["amount", "account_age_days", "tx_last_hour",
             "receiver_tx_count_24h", "sender_daily_amount_sum",
             "amount_to_balance_ratio", "days_since_last_tx",
             "fraud_pattern"]

print(ml_catches_rule_misses[disp_cols].head(10))

# Rule catches, ML misses (rule better here)
rule_catches_ml_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 1) &
    (test_df["ml_pred"] == 0)
]
print(f"\nRules caught fraud that ML missed: {len(rule_catches_ml_misses)}")
print(rule_catches_ml_misses[disp_cols].head(10))

# Both missed (most dangerous — neither system reacted)
both_miss = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 0)
]
print(f"\nBoth missed: {len(both_miss)}")
print(both_miss[disp_cols].head(10))

ML caught fraud that rules missed: 44
Rules caught fraud that ML missed: 11
Both caught: 42
Both missed: 28


## Conclusions (actual numbers from v4 run)

### Comparison summary (test set, 4000 rows, 125 fraud)

| Category | Count |
|---|---|
| Both caught (ML+Rules agree on fraud) | 42 |
| ML better (rules miss, ML catches) | 44 |
| Rules better (ML miss, rules catch) | 11 |
| Both missed | 28 |

**Key change from v3:** With realistic balances (median ~27k vs ~3.6k),
the `insufficient_balance` rule fires much less often, reducing rule
recall from 72% to 42%. ML is more robust — it learned to use
`amount_to_balance_ratio` (now 2nd most important feature).

### Recall by fraud pattern

| Pattern | Rules recall | ML recall |
|---|---|---|
| new_account_fraud | 80% | 100% |
| structuring_fraud | 86% | 89% |
| velocity_fraud | 100% | 80% |
| dormant_reactivation_fraud | 75% | 38% |
| mule_fraud | 23% | 73% |
| balance_drain_fraud | 0% | 26% |

### Feature importances

| Feature | Importance |
|---|---|
| amount | 0.404 |
| amount_to_balance_ratio | 0.228 |
| days_since_last_tx | 0.114 |
| account_age_days | 0.086 |
| receiver_tx_count_24h | 0.061 |
| sender_daily_amount_sum | 0.051 |
| transaction_hour | 0.043 |
| tx_last_hour | 0.015 |

### Key takeaways
- Balance fix made `amount_to_balance_ratio` a meaningful feature (2nd highest importance).
- ML strongly outperforms rules on mule_fraud (73% vs 23%) — ML uses amount_to_balance_ratio
  and days_since_last_tx to detect mules even when amounts are normal.
- balance_drain_fraud remains the hardest pattern for both systems.
- `transaction_hour` is 7th at 0.043 (no leak).
- ROC-AUC: 0.9623 (improved from 0.947 on v3).